# W08_M3.1_产品表面缺陷尺寸 —— 教师版·分步执行（ABCD 全层完成态）

> 形态：与课堂分步一致，每步含【参考答案】与真实执行输出；非聚合脚本。


# AI应用基础 · 工单M3.1《产品表面缺陷尺寸测量系统》
**第8周 | 2课时(80分钟) | 理实一体**  | 教师参考版

| 层 | 任务定位 | 达标要求 |
|----|---------|---------|
| **A 师傅** | 进阶拓展 | 多缺陷件全检出+误检率归零，并分析每个误检的成因；示范讲解 |
| **B 组长** | 标准达成 | 误差<5%；多缺陷全检出且美观 |
| **C 操作员** | 基础完成 | 正确检出标注；误差<10% |
| **D 学徒** | 跟做保底 | 跟做示例跑通TODO-1, 程序出结果 |
> 每4周按工单成绩动态升降层（连续2次≥B升一层）

## 一、课前资讯（Predict·预测）
> **预习快问**：像素面积350px²、当量0.1mm/px，物理面积是多少mm²？

> **课前检查**：导学单+预习自测已完成；data/数据已就位、starter可运行（--check环境OK）

## 二、PRIMM五步法（先读后写）
本工单按国际编程教学法 **PRIMM** 组织：
1. **Predict 预测**——先运行下方示例，猜输出再验证；
2. **Run 运行**——跑通环境与数据；
3. **Investigate 研究**——读懂starter每段在做什么；
4. **Modify 修改**——完成TODO改造；
5. **Make 创造**——独立完成工单任务。


In [1]:
# --- 环境与数据准备(勿改, 运行即可) ---
import os, sys, subprocess
os.chdir(os.path.abspath("../"))          # 切到课程包根目录(data/所在处)
print("工作目录:", os.getcwd())

subprocess.run([sys.executable, "数据与工具/generate_data.py", "m3"], check=True)
print("数据已生成: data/")

工作目录: /Users/joelgong/PycharmProjects/ai-course-teaching-resources/01-AI应用基础


M3.1 -> data/surface_parts 40 件(ground_truth.json为教师核对用, 勿下发)
完成。数据输出目录: /Users/joelgong/PycharmProjects/ai-course-teaching-resources/01-AI应用基础/data
数据已生成: data/


## 三、任务要求（Make）
**实施步骤**
1. 灰度化→滤波→阈值分割→开运算
2. findContours提取轮廓并过滤噪点
3. 判型(划痕/凹坑)并按当量换算尺寸
4. 画框+文字标注输出结果图

**验收标准**：及格(D)=`正确检出标注；误差<10%` ；优秀(B+)=`误差<5%；多缺陷全检出且美观`


### Starter 1：`# -*- coding: utf-8 -*-`
（Investigate: 读代码→Modify: 补全TODO）

In [2]:
# -*- coding: utf-8 -*-
"""M3.1 产品表面缺陷尺寸测量 —— 学生代码框架
像素当量: 0.1 mm/px (已标定)
"""
import glob

import cv2
import numpy as np

PX_TO_MM = 0.1
PARTS = sorted(glob.glob("data/surface_parts/part_*.png"))

### Starter 2：`def measure(path):`
（Investigate: 读代码→Modify: 补全TODO）

In [3]:
def measure(path):
    """返回 标注图, [(类型, 尺寸mm)]"""
    img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
    blur = cv2.GaussianBlur(img, (5, 5), 0)
    _, binary = cv2.threshold(blur, 0, 255,
                              cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
    binary = cv2.morphologyEx(binary, cv2.MORPH_OPEN, np.ones((3, 3), np.uint8))
    contours, _ = cv2.findContours(binary, cv2.RETR_EXTERNAL,
                                   cv2.CHAIN_APPROX_SIMPLE)
    results = []
    anno = cv2.cvtColor(img, cv2.COLOR_GRAY2BGR)
    for cnt in contours:
        if cv2.contourArea(cnt) < 15:
            continue
        rect = cv2.minAreaRect(cnt)
        w, h = sorted(rect[1])
        if w / max(h, 1) > 3:
            txt = f"scratch {w * PX_TO_MM:.2f}mm"
            results.append(("划痕", round(w * PX_TO_MM, 2)))
        else:
            txt = f"pit {cv2.contourArea(cnt) * PX_TO_MM ** 2:.2f}mm2"
            results.append(("凹坑", round(cv2.contourArea(cnt) * PX_TO_MM ** 2, 2)))
        box = cv2.boxPoints(rect).astype(int)
        cv2.drawContours(anno, [box], 0, (0, 0, 255), 2)
        cv2.putText(anno, txt, (int(box[:, 0].min()), int(box[:, 1].min()) - 6),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.45, (0, 0, 255), 1)
    return anno, results

### 💡 分层提示（先自己试5分钟再看）
**思路提示**
> - TODO-1 预处理: 高斯模糊(5×5) → 阈值分割(先试100,
>   再试Otsu) → 开运算去噪点
>   （在此处编写你的代码——依据上方注释/文档字符串）
> - TODO-2 轮廓: cv2.findContours(binary,
>   cv2.RETR_EXTERNAL, ...)
> - TODO-3 判型: 最小外接矩形长宽比>3→划痕(长度=长边),
>   否则→凹坑(面积)
> - （在此处编写你的代码——依据上方注释/文档字符串） TODO-4 换算:
>   长度mm = 长边px * PX_TO_MM
> - TODO-5 标注: cv2.drawContours +
>   cv2.putText(类型+尺寸)
> - TODO-6 决策矩阵: 全局100 / Otsu / 自适应
>   三种阈值各测5件, 记录检出与误检

| 层 | 这样做 |
|----|--------|
| **D 学徒** | 跟教师演示逐步照做：定位TODO行 → 按上面思路逐句翻译成代码 → 运行看输出 → 对不上就看下方"报错自查" |
| **C 操作员** | 只看思路提示独立实现；卡住再对照教师课堂演示 |
| **B/A 组长·师傅** | 不看提示直接做；完成后做扩展：改参数/换实现对比效果，写进工单"参数-收益"留痕 |

**报错自查三板斧**：①读最后一行错误信息 → ②检查变量名/缩进/括号 → ③还不行问同层同学或教师
**自检**：对照上方"验收标准"，及格(D)=`正确检出标注；误差<10%`，优秀(B+)=`误差<5%；多缺陷全检出且美观`

### Starter 3：`def main():`
（Investigate: 读代码→Modify: 补全TODO）

In [4]:
def main():
    for p in PARTS[:5]:
        anno, res = measure(p)
        out = "result_" + p.split("/")[-1]
        cv2.imwrite(out, anno)
        print(out, "→", res)

    # TODO-6 决策矩阵: 全局100 / Otsu / 自适应 三种阈值各测5件, 记录检出与误检

### Starter 4：`if __name__ == "__main__":`
（Investigate: 读代码→Modify: 补全TODO）

In [5]:
if __name__ == "__main__":
    main()

result_part_000.png → [('凹坑', 2.24)]
result_part_001.png → [('凹坑', 3.28)]
result_part_002.png → [('凹坑', 2.77)]
result_part_003.png → [('凹坑', 8.48)]
result_part_004.png → [('凹坑', 2.24)]


## 四、提交要求与避坑
- 提交: `学号_姓名_M3.1.py` 或本notebook(含输出)
- ⚠ 面积换算记得平方：350px²→3.5mm²
- ⚠ 轮廓用RETR_EXTERNAL避免内轮廓重复计数
- ⚠ 阈值方向：缺陷暗背景亮→用THRESH_BINARY_INV

---

## 教师参考答案（来源：`教师参考/solutions.py` · `m31()`）

> **发放纪律：本附录勿随学生版下发。**

上方学生版 TODO 与下方参考实现的对应关系：

| 学生版函数 | 说明 |
|---|---|
| `main()` | 见下方 `m31()` 内对应逻辑 |

In [6]:
def m31():
    import cv2

    PX = 0.1
    for i in range(3):
        p = f"data/surface_parts/part_{i:03d}.png"
        img = cv2.imread(p, cv2.IMREAD_GRAYSCALE)
        blur = cv2.GaussianBlur(img, (5, 5), 0)
        _, bw = cv2.threshold(blur, 0, 255,
                              cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
        bw = cv2.morphologyEx(bw, cv2.MORPH_OPEN,
                              np.ones((3, 3), np.uint8))
        cnts, _ = cv2.findContours(bw, cv2.RETR_EXTERNAL,
                                   cv2.CHAIN_APPROX_SIMPLE)
        anno = cv2.cvtColor(img, cv2.COLOR_GRAY2BGR)
        for c in cnts:
            if cv2.contourArea(c) < 15:
                continue
            rect = cv2.minAreaRect(c)
            w, h = sorted(rect[1])
            if w / max(h, 1) > 3:                      # 划痕
                val = w * PX
                txt = f"scratch {val:.2f}mm"
            else:                                      # 凹坑
                val = cv2.contourArea(c) * PX ** 2
                txt = f"pit {val:.2f}mm2"
            box = cv2.boxPoints(rect).astype(int)
            cv2.drawContours(anno, [box], 0, (0, 0, 255), 2)
            cv2.putText(anno, txt, (box[:, 0].min(), box[:, 1].min() - 6),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.45, (0, 0, 255), 1)
            print(p.split("/")[-1], txt)
        cv2.imwrite(f"result_{i:03d}.png", anno)

## 运行与验证

上方参考实现可直接运行（合成数据、CPU 离线、结果可复现）。
多解法对比见《多解法手册.md》。

## A·师傅层：多解对比与帮带要点

## 运行与验证

上方参考实现可直接运行（合成数据、CPU 离线、结果可复现）。
多解法对比见《多解法手册.md》。

---

## A·师傅层：多解对比与帮带要点

---

## 运行与验证

上方参考实现可直接运行（合成数据、CPU 离线、结果可复现）。
多解法对比见《多解法手册.md》。

---

## A·师傅层：多解对比与帮带要点

---

## 执行与验证

---

## ABCD 验收对照

- **D·及格/C·良好**：TODO 分步单元（上方）逐个执行通过，输出即证据（0 个参考答案单元）。
- **B·优秀**：标注「(优秀)」的扩展单元已实现并执行。
- **A·师傅**：多解互验与帮带要点见上节；可用于 2 分钟费曼展示。

---

## ABCD 验收对照

- **D·及格/C·良好**：TODO 分步单元（上方）逐个执行通过，输出即证据（0 个参考答案单元）。
- **B·优秀**：标注「(优秀)」的扩展单元已实现并执行。
- **A·师傅**：多解互验与帮带要点见上节；可用于 2 分钟费曼展示。

## ABCD 验收对照

- **D·及格/C·良好**：TODO 分步单元（上方）逐个执行通过，输出即证据（0 个参考答案单元）。
- **B·优秀**：标注「(优秀)」的扩展单元已实现并执行。
- **A·师傅**：多解互验与帮带要点见上节；可用于 2 分钟费曼展示。
